# R07 · Frozen evaluators on fresh queries

Freeze existing evaluators and test new query geometry/groups. Generalization failures remain part of the evidence that motivates R08.

**One decision:** does the SA / raw2-proxy / frozen-ML tradeoff from 16 persist on newly generated queries, without
changing the original histogram answer, model, feature formulas or acceptance rule? No model is trained or selected.
The three methods evaluate exactly the same H for every new query. There is no 5% accepted-risk target or new gate.

## Run on Databricks

1. Pull this branch in your personal Git folder and open **R07** on the same runtime used for R05/R06.
2. Run the configuration cell. Normally **clear `run_dir` and leave it blank** to select the latest completed
   **16 `information-budget-...`** result, by modification time only. To reproduce one source, enter that complete
   directory. Do not enter 15's `within-source-...`, a notebook path or ZIP.
3. Leave `seeds=0,1,2`. These identify SAVED model seeds, not random seeds for generating queries. Leave
   **`query_seed=20261116`** and **`replication_cells=1000`** fixed for the planned validation. Choose settings before
   viewing new labels/results; do not search query salts for a good outcome.
4. A small implementation check may use `seeds=0` and a smaller `replication_cells`; the primary cohort still contains
   ALL globally unused source cells. To run the planned comparison, restore seeds 0,1,2 and replication_cells=1000.
   Reusing the same query seed produces the same geometry, not another independent replication.
5. Normally leave `raw_data_dir`, `queries_file` and `logger_file` blank. Existing source R05/08d files, saved models,
   raw2 payloads and matching raw points must remain accessible. Relocation overrides still require matching hashes.
6. Run All. If needed, the dependency cell installs the source run's LightGBM version. Other runtime packages must match.
   **No old notebook training/preparation entry point is executed.** New exact labels are computed from raw points.
7. Share **`fresh_query_validation_results.zip`** from the printed new `fresh-query-validation-...` directory.
   Previous results are preserved. A failed run also exports its available diagnostics in the same ZIP format.

## Predeclared cohorts, kept separate

| Cohort | Geometry / membership | Claim scope |
|---|---|---|
| `new_groups` (PRIMARY) | One new query in EVERY source generating-cell ID absent from ALL original query rectangles, including original truth<10/zero cases. Evaluate with all requested saved models. | New generating IDs and new labels on the same raw dataset |
| `existing_test_groups` (SUPPLEMENT) | Up to 1000 original outer-test cells per model seed, fixed hash sampling, one new query/cell; only that seed's held-out groups are scored. | Replication within groups already examined earlier |

The observed authors' workload uses a one-source-cell rectangle shifted diagonally. This protocol defines a new member
with equal x/y fractional shift U(0,sqrt(.05)), deterministically keyed by cell and query_seed. Geometry is compared with
ALL old rectangles and near-duplicates rejected. The protocol, geometry and memberships are hashed BEFORE reading new truth.
Every source cell's candidate is preallocated in fixed order, so geometry does not change when requesting more model seeds.

The current inputs have 8064 possible source cells and 6757 groups represented by old queries, leaving **1307 primary
candidates**. Exact counts below the inherited minimum10 are excluded and reported, **never replaced to fill a quota**.
Zero-count performance is not claimed. Candidate counts are not assumed to be eligible counts.

**This is not a new dataset or strict non-overlapping geographic holdout.** Different source grids can overlap, and the
synopses already summarize the same raw data. A new-query test improves the evidence beyond repeated old-test inspection,
but does not erase all study-selection effects or establish universal generalization.

## Frozen evaluation

- Original SA: faithful live formula, numerical relative-error MAE undefined.
- Raw2 proxy: `abs(H-Sfine)/max(Sfine,1)`, exactly as 16; no learned mapping or correction.
- Raw2 ML: exact saved R05 model files, 21 features, direct relative-error output floored at zero, never exponentiated.

Full-cohort MAE/P90/bias/AUC and complete-tie 10/25/50/100% ranking diagnostics are reported for each cohort/seed/rate.
No test label selects a model, formula, salt or gate. Runtime is measured on common hash-selected queries, with real
estimate/score/inference, randomized arm order and post-timer parity. Resident costs exclude raw reads and exact fallback;
new truth generation/indexing is offline validation cost, not a proposed production exact-query speedup.

In [ ]:
import ast
import base64
import gzip
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd
from scipy.special import betainc

EPSILON = .25
FRACTIONS = (.10, .25, .50, 1.)
METHODS = ('base_sa', 'raw2_proxy', 'raw2_ml')
FAMILIES = ('new_groups', 'existing_test_groups')
MIN_EXACT = 10
IMPLEMENTATION_SHA256 = 'a24658c7717bd9e1cf047d374dbcf0853ef198c31dd0f7dd606ffcd08ce9b0fe'


def notebook_defaults():
    try:
        ctx=dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root=ctx.notebookPath().get().split('/ml/notebooks/')[0]
        repo=root if root.startswith('/Workspace/') else '/Workspace'+root
        output='/Workspace/Users/'+ctx.userName().get()+'/synopsislake-results'
    except (NameError,AttributeError):
        repo,output=str(Path.cwd()),str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo,output_root=output,run_dir='',raw_data_dir='',queries_file='',logger_file='',
                seeds='0,1,2',query_seed='20261116',replication_cells='1000',timing_queries='30',timing_repeats='5')


DEFAULTS=notebook_defaults()
try:
    for key,value in DEFAULTS.items():dbutils.widgets.text(key,value)
    get_config=dbutils.widgets.get
except NameError:
    def get_config(key):return os.environ.get(key.upper(),DEFAULTS[key])


def load_settings():
    cfg={k:str(get_config(k)).strip() for k in DEFAULTS}
    cfg['seeds']=[int(x) for x in cfg['seeds'].split(',')]
    if not cfg['seeds'] or len(set(cfg['seeds']))!=len(cfg['seeds']):raise ValueError('Use distinct saved model seeds')
    for key in ('query_seed','replication_cells','timing_queries','timing_repeats'):
        cfg[key]=int(cfg[key])
    if min(cfg['replication_cells'],cfg['timing_queries'],cfg['timing_repeats'])<1:
        raise ValueError('Replication and timing settings must be positive')
    return cfg


def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def save_json(path,value):
    path=Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2,allow_nan=False,default=str));temporary.replace(path)


def stable_ids(ids,n,salt):
    return sorted(sorted(set(map(str,ids)),key=lambda x:hashlib.sha256(f'{salt}:{x}'.encode()).digest())[:n])


def resolve_source(cfg):
    if cfg['run_dir']:source=Path(cfg['run_dir']).expanduser().resolve()
    else:
        options=[]
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info=json.loads(path.read_text())
            if (info.get('status')=='complete' and info.get('study')=='information-budget-comparison'
                    and set(cfg['seeds'])<=set(info.get('config',{}).get('seeds',[]))):options.append(path)
        if not options:raise FileNotFoundError('Set run_dir to a completed notebook 16 information-budget-... result directory, not 15 or a ZIP')
        source=max(options,key=lambda p:p.stat().st_mtime_ns).parent
    prior=json.loads((source/'run.json').read_text())
    if (prior.get('status')!='complete' or prior.get('study')!='information-budget-comparison'
            or not set(cfg['seeds'])<=set(prior['config']['seeds'])):
        raise ValueError('run_dir must be a completed notebook 16 run containing the requested model seeds')
    print('Frozen source16:',source,flush=True)
    return source,prior

In [ ]:
# INSTALL_DEPENDENCIES
import importlib.util
_config_for_install = load_settings()
_source_for_install, _prior_for_install = resolve_source(_config_for_install)
if importlib.util.find_spec('lightgbm') is None:
    get_ipython().run_line_magic('pip', 'install lightgbm==' + _prior_for_install['package_versions']['lightgbm'])
else:
    print('LightGBM is installed; the frozen runtime versions will be verified.')

## Reuse without retraining

Only hash-pinned definitions from R06/R05/R03/R02 are loaded. Matching source15 model/payload/feature identities must equal
those frozen by16. The same runtime versions and original input hashes are checked. No call to fit, training entry point,
feature search, gate calibration or finer-grid construction occurs in this notebook.

In [ ]:
REUSE_SPECS_16 = {'information-budget-04': None, 'information-budget-06': ['_sa_hist_parts', 'sa_alpha', '_sa_overlap', '_sa_query_overlap', 'histogram_count', 'query_answer_sa'], 'information-budget-08': ['verified_path', 'load_inputs', 'load_export_hist'], 'information-budget-12': ['proxy_error', 'fixed_coverage'], 'information-budget-14': ['runtime_call']}
REUSE_HASHES_16 = {'information-budget-04': '133cafa3870643c131f4cb336a3b2775ccc829c2ebd8e9120d58fbc42482f5e6', 'information-budget-06': 'e3805d6e30a42576fcb4052450f7d985c9830de82d73e38ef52bb9f0b06f10fc', 'information-budget-08': 'ddc98ddb2d0f14f9437f99b0c26075add8e4086cef870e73f587282a8fe0bbd3', 'information-budget-12': '0245ba5f06b2549e1180b02c1837b74a46b6c3191c960e1b66c2945c6e605f39', 'information-budget-14': 'd953057ef1b5fe703e52ea741f5fe51724bc3b0b8a592f26cb23fd0272f9f7a9'}


def load_verified16(repo):
    path=Path(repo).resolve()/'ml/notebooks/18_information_budget_comparison.ipynb'
    try:
        raw=path.read_bytes();doc=json.loads(raw);method='workspace filesystem'
    except (OSError,ValueError,UnicodeError):
        if not str(path).startswith('/Workspace/'):raise
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ExportFormat
        client=WorkspaceClient();last=None
        for candidate in (str(path)[len('/Workspace'):],str(path)[len('/Workspace'):].removesuffix('.ipynb')):
            try:
                raw=base64.b64decode(client.workspace.export(path=candidate,format=ExportFormat.JUPYTER).content)
                doc=json.loads(raw);method='Workspace API using runtime identity';break
            except Exception as error:last=error
        else:raise ValueError('Cannot read matching notebook 16: '+str(last)) from last
    env=dict(globals());env['__name__']='verified_notebook16_definitions'
    for label,names in REUSE_SPECS_16.items():
        matches=[]
        for cell in doc['cells']:
            if cell['cell_type']!='code':continue
            try:tree=ast.parse(''.join(cell['source']))
            except SyntaxError:continue
            if names is not None:
                nodes=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in names]
                if {n.name for n in nodes}!=set(names):continue
                tree=ast.Module(body=nodes,type_ignores=[])
            elif any(not isinstance(n,(ast.Import,ast.ImportFrom,ast.Assign,ast.FunctionDef)) for n in tree.body):continue
            if hashlib.sha256(ast.dump(tree,include_attributes=False).encode()).hexdigest()==REUSE_HASHES_16[label]:matches.append(tree)
        if len(matches)!=1:raise ValueError('Notebook16 definition changed/missing: '+label+'. Pull matching notebooks 12-17')
        exec(compile(matches[0],str(path)+':'+label,'exec'),env)
    helpers,namespace,parents=env['load_verified_helpers'](repo)
    return env,helpers,namespace,dict(notebook16=dict(path=str(path),read_method=method,document_sha256=hashlib.sha256(raw).hexdigest(),
        definition_ast_sha256=REUSE_HASHES_16),parents=parents)


def load_frozen_inputs(cfg,source16,prior16,destination,state):
    env,helpers,namespace,reuse=load_verified16(cfg['repo_root'])
    source15=Path(prior16['source15_run'])
    env['verified_path'](source15/'run.json',prior16['source15_manifest_sha256'],'source15 manifest frozen by 16')
    prior15=json.loads((source15/'run.json').read_text())
    for name,digest in prior16['source15_artifact_sha256'].items():
        env['verified_path'](source15/name,digest,'source15 artifact frozen by16')
    data=env['load_inputs'](cfg,source15,prior15,destination,helpers,namespace,state)
    # Old test scores/truth are not query-generation inputs. Their only purpose in load_inputs is provenance reuse.
    state.update(source16_run=str(source16),source16_manifest_sha256=sha(source16/'run.json'),reuse=reuse,
                 source15_manifest_sha256=sha(source15/'run.json'),num_threads=int(prior15['inherited_training_config']['num_threads']))
    return data,env,helpers,namespace,prior15

## Freeze query geometry before labels

The generator receives only old rectangles, source grid boundaries, outer role IDs and fixed configuration. Old true
counts and model scores are not arguments. It excludes primary IDs observed in ANY old rectangle, not just eligible
old queries. Secondary memberships cannot overlap the corresponding model's fitting/calibration groups.

The new family is explicitly defined rather than claiming bit-for-bit reconstruction of an unavailable original
generator. `generation_audit.json` records whether the old geometry matches one-cell diagonal shifts. Query coordinates
are stored with17 significant digits; downstream readers of NEW plan/label CSVs should use `float_precision='round_trip'`.
Original inputs retain their original parser for compatibility. Do not round new coordinates before labeling/inference.

In [ ]:
"""Frozen, geometry-only query generation for notebook 17.

No function in this module receives labels, model predictions or acceptance
outcomes. Coarse count arrays are used only to validate histogram dimensions.
"""
import hashlib
import math
from itertools import product

import numpy as np
import pandas as pd

GENERATION_VERSION = '17-fresh-diagonal-v1'
MAX_DIAGONAL_SHIFT = math.sqrt(.05)
NEAR_GEOMETRY_TOL = 1e-10
MAX_GEOMETRY_ATTEMPTS = 256
PLAN_COLUMNS = ['query_id','family','cell','source_id','cell_x','cell_y',
                'x0','x1','y0','y1','shift','attempt']
MEMBERSHIP_COLUMNS = ['query_id','family','cell','seed']


def _ordered_grids(coarse_hists, source_ids):
    if len(coarse_hists) != len(source_ids) or not len(coarse_hists):
        raise ValueError('Each coarse histogram requires one distinct source ID')
    ids = []
    for value in source_ids:
        try:
            number = float(value)
        except (TypeError,ValueError) as error:
            raise ValueError('Source IDs must be distinct nonnegative integers') from error
        if not np.isfinite(number) or int(number) != number or int(number) < 0:
            raise ValueError('Source IDs must be distinct nonnegative integers')
        ids.append(int(number))
    if len(set(ids)) != len(ids):
        raise ValueError('Duplicate source IDs')
    ordered = []
    for source_id, hist in sorted(zip(ids,coarse_hists),key=lambda item:item[0]):
        if len(hist) != 4:
            raise ValueError('Histogram must be (rate, bx, by, counts)')
        bx,by = np.asarray(hist[1],float),np.asarray(hist[2],float)
        if (bx.ndim != 1 or by.ndim != 1 or len(bx) < 2 or len(by) < 2 or
                not np.isfinite(bx).all() or not np.isfinite(by).all() or
                (np.diff(bx) <= 0).any() or (np.diff(by) <= 0).any() or
                np.shape(hist[3]) != (len(bx)-1,len(by)-1)):
            raise ValueError('Invalid source boundaries or dimensions')
        width,height = float(np.diff(bx).mean()),float(np.diff(by).mean())
        if (not np.allclose(np.diff(bx),width,rtol=1e-8,atol=1e-12) or
                not np.allclose(np.diff(by),height,rtol=1e-8,atol=1e-12)):
            raise ValueError('Fresh queries require the original uniform source grids')
        ordered.append(dict(source_id=source_id,bx=bx,by=by,width=width,height=height))
    return ordered


def _rectangles(rectangles):
    values = np.asarray(rectangles,float)
    if (values.ndim != 2 or values.shape[1] != 4 or not np.isfinite(values).all() or
            (values[:,0] >= values[:,1]).any() or (values[:,2] >= values[:,3]).any()):
        raise ValueError('Rectangles must be an N x 4 finite increasing array')
    return values


def _infer_with_grids(rectangles, grids):
    q = _rectangles(rectangles)
    if not len(q):
        return np.empty(0,dtype=str)
    width,height = q[:,1]-q[:,0],q[:,3]-q[:,2]
    centres_x,centres_y = (q[:,0]+q[:,1])/2,(q[:,2]+q[:,3])/2
    size_gap = np.asarray([[abs(w-g['width'])+abs(h-g['height']) for g in grids]
                           for w,h in zip(width,height)])
    source_position = size_gap.argmin(axis=1)
    # This intentionally retains the inherited searchsorted(side='left') - 1
    # convention; do not silently replace it with a new grouping definition.
    i = np.asarray([np.searchsorted(grids[s]['bx'],x)-1 for s,x in zip(source_position,centres_x)])
    j = np.asarray([np.searchsorted(grids[s]['by'],y)-1 for s,y in zip(source_position,centres_y)])
    return np.asarray([f"{grids[s]['source_id']}_{a}_{b}" for s,a,b in zip(source_position,i,j)],dtype=str)


def infer_cells(rectangles, coarse_hists, source_ids):
    """Match model_study.source_cells on numerically sorted source file IDs."""
    return _infer_with_grids(rectangles,_ordered_grids(coarse_hists,source_ids))


def _geometry_hash(rectangle):
    # Little-endian float64 geometry, with signed zeros canonicalized.
    values = np.asarray(rectangle,dtype='<f8').copy()
    values[values == 0] = 0.
    return hashlib.sha256(values.tobytes()).hexdigest()


def _shift_for_cell(query_seed, cell, attempt):
    salt = f'{GENERATION_VERSION}:{query_seed}:{cell}:{attempt}'
    integer = int.from_bytes(hashlib.sha256(salt.encode()).digest()[:8],'big') >> 11
    unit = (integer+.5)/(2**53)
    unit = min(float(np.nextafter(1.,0.)),max(float(np.nextafter(0.,1.)),float(unit)))
    return MAX_DIAGONAL_SHIFT*unit


def _old_family_diagnostics(q, cells, grid_by_id):
    if not len(q):
        return dict(query_count=0,one_cell_size_fraction=None,diagonal_shift_fraction=None,
                    shift_range_fraction=None,max_xy_shift_difference=None)
    sizes,diagonal,ranges,differences = [],[],[],[]
    for rectangle,cell in zip(q,cells):
        sid,i,j = map(int,cell.split('_'));g=grid_by_id[sid]
        valid = 0 <= i < len(g['bx'])-1 and 0 <= j < len(g['by'])-1
        if not valid:
            sizes.append(False);diagonal.append(False);ranges.append(False);continue
        dx,dy = g['bx'][i+1]-g['bx'][i],g['by'][j+1]-g['by'][j]
        ox,oy = (rectangle[0]-g['bx'][i])/dx,(rectangle[2]-g['by'][j])/dy
        sizes.append(bool(np.isclose((rectangle[1]-rectangle[0])/dx,1.,rtol=0,atol=1e-8) and
                          np.isclose((rectangle[3]-rectangle[2])/dy,1.,rtol=0,atol=1e-8)))
        differences.append(float(abs(ox-oy)))
        diagonal.append(bool(abs(ox-oy) <= 1e-8))
        ranges.append(bool(-1e-8 <= ox <= MAX_DIAGONAL_SHIFT+1e-8 and
                           -1e-8 <= oy <= MAX_DIAGONAL_SHIFT+1e-8))
    return dict(query_count=len(q),one_cell_size_fraction=float(np.mean(sizes)),
                diagonal_shift_fraction=float(np.mean(diagonal)),shift_range_fraction=float(np.mean(ranges)),
                max_xy_shift_difference=max(differences) if differences else None)


def generate_plan(old_rectangles, coarse_hists, source_ids, outer, seeds,
                  query_seed=20261116, replication_cells=1000):
    """Return a frozen query plan, seed membership and geometry-only audit.

    All old rectangles must be supplied, including rows later excluded by the
    exact-count cohort. No candidate is selected or replaced using new truth.
    A geometry is preallocated for EVERY possible source cell in a fixed order,
    making collision handling independent of which model seeds are requested.
    """
    q = _rectangles(old_rectangles)
    grids = _ordered_grids(coarse_hists,source_ids)
    if (not seeds or len(set(seeds)) != len(seeds) or
            any(not np.isfinite(s) or int(s) != s for s in seeds)):
        raise ValueError('Use distinct integer model seeds')
    seeds = sorted(map(int,seeds))
    if (not np.isfinite(query_seed) or int(query_seed) != query_seed or
            not np.isfinite(replication_cells) or int(replication_cells) != replication_cells or replication_cells < 0):
        raise ValueError('query_seed must be integer; replication_cells must be a nonnegative integer')
    query_seed,replication_cells = int(query_seed),int(replication_cells)
    grid_by_id = {g['source_id']:g for g in grids}
    inventory = []
    for g in grids:
        for i in range(len(g['bx'])-1):
            for j in range(len(g['by'])-1):
                inventory.append((f"{g['source_id']}_{i}_{j}",g['source_id'],i,j))
    possible = {cell for cell,_,_,_ in inventory}
    old_cell_array = _infer_with_grids(q,grids)
    old_cells = set(old_cell_array)
    grouped_old = {}
    for cell,rectangle in zip(old_cell_array,q):
        grouped_old.setdefault(cell,[]).append(rectangle)
    grouped_old = {cell:np.asarray(values,float) for cell,values in grouped_old.items()}
    old_hashes = {_geometry_hash(rectangle) for rectangle in q}
    if not {'seed','cell','split'} <= set(outer.columns):
        raise ValueError('Outer membership requires seed, cell and split')
    outside = outer.copy()
    if outside[['seed','cell','split']].isna().any().any():
        raise ValueError('Missing outer role identifiers')
    if not np.equal(outside.seed.to_numpy(float),np.floor(outside.seed.to_numpy(float))).all():
        raise ValueError('Outer model seeds must be integers')
    outside['seed'] = outside.seed.astype(int)
    outside['cell'] = outside.cell.astype(str)
    outside = outside[outside.seed.isin(seeds)]
    selected_secondary = {}
    for seed in seeds:
        part = outside[outside.seed == seed]
        if set(part.split) != {'train','calibration','test'}:
            raise ValueError('Each requested seed needs train, calibration and test roles')
        if not set(part.cell) <= possible:
            raise ValueError('Outer membership contains invalid source-cell IDs')
        if not set(part.cell) <= old_cells:
            raise ValueError('Outer cell IDs disagree with all-old query geometry')
        roles = part[['cell','split']].drop_duplicates()
        if roles.cell.duplicated().any():
            raise ValueError('A source cell overlaps outer train/calibration/test roles')
        test_cells = set(roles.loc[roles.split == 'test','cell'])
        ordered = sorted(test_cells,key=lambda cell:hashlib.sha256(
            f'{GENERATION_VERSION}:replication:{query_seed}:{cell}'.encode()).digest())
        selected_secondary[seed] = set(ordered[:replication_cells])
    primary = possible-old_cells
    if not primary:
        raise ValueError('No globally unused source-cell groups; primary validation cannot be substituted')
    secondary = set().union(*selected_secondary.values())
    if primary & secondary:
        raise ValueError('Primary and secondary populations overlap')
    generated = {};new_hashes=set();rejected_near_old=0;rejected_duplicate=0
    new_geometry_bins = {}
    bucket_scale = NEAR_GEOMETRY_TOL*np.asarray([max(g['width'] for g in grids)]*2 +
                                               [max(g['height'] for g in grids)]*2)
    neighbor_offsets = list(product((-1,0,1),repeat=4))
    # Preallocating all possible cells also verifies the inherited grouping can
    # identify each source grid; indistinguishable source cell sizes fail here.
    for cell,sid,i,j in inventory:
        g=grid_by_id[sid];dx=g['bx'][i+1]-g['bx'][i];dy=g['by'][j+1]-g['by'][j]
        old=grouped_old.get(cell)
        scale=np.asarray([dx,dx,dy,dy])
        for attempt in range(MAX_GEOMETRY_ATTEMPTS):
            shift=_shift_for_cell(query_seed,cell,attempt)
            x0=g['bx'][i]+shift*dx;y0=g['by'][j]+shift*dy
            rectangle=np.asarray([x0,x0+dx,y0,y0+dy],float)
            digest=_geometry_hash(rectangle)
            if digest in old_hashes or (old is not None and
                    np.any(np.max(np.abs((old-rectangle)/scale),axis=1) <= NEAR_GEOMETRY_TOL)):
                rejected_near_old+=1;continue
            bucket = tuple(int(np.floor(value/unit)) for value,unit in zip(rectangle,bucket_scale))
            near_new = False
            for delta in neighbor_offsets:
                key = tuple(a+b for a,b in zip(bucket,delta))
                for previous in new_geometry_bins.get(key,[]):
                    if np.max(np.abs((previous-rectangle)/scale)) <= NEAR_GEOMETRY_TOL:
                        near_new = True;break
                if near_new:break
            if digest in new_hashes or near_new:
                rejected_duplicate+=1;continue
            inferred=_infer_with_grids(rectangle.reshape(1,4),grids)[0]
            if inferred != cell:
                raise ValueError('Source grids cannot reproduce declared generating-cell IDs; check ambiguous source sizes or boundaries')
            generated[cell]=dict(cell=cell,source_id=sid,cell_x=i,cell_y=j,
                x0=float(rectangle[0]),x1=float(rectangle[1]),y0=float(rectangle[2]),y1=float(rectangle[3]),
                shift=float(shift),attempt=attempt,geometry_hash=digest)
            new_hashes.add(digest)
            new_geometry_bins.setdefault(bucket,[]).append(rectangle)
            break
        else:
            raise ValueError('Cannot generate distinct finite geometry for cell '+cell)
    plans=[];members=[]
    for family,chosen in (('new_groups',primary),('existing_test_groups',secondary)):
        for cell,_,_,_ in inventory:
            if cell not in chosen:continue
            geometry=generated[cell]
            query_id=f"q17-{family}-{cell}-{geometry['geometry_hash'][:16]}"
            row=dict(query_id=query_id,family=family,**{k:v for k,v in geometry.items() if k!='geometry_hash'})
            plans.append(row)
            for seed in seeds:
                if family=='new_groups' or cell in selected_secondary[seed]:
                    members.append(dict(query_id=query_id,family=family,cell=cell,seed=seed))
    plan=pd.DataFrame(plans,columns=PLAN_COLUMNS)
    membership=pd.DataFrame(members,columns=MEMBERSHIP_COLUMNS)
    if plan.query_id.duplicated().any() or membership.duplicated(['query_id','seed']).any():
        raise ValueError('Duplicate planned query identities')
    if plan[['x0','x1','y0','y1']].duplicated().any():
        raise ValueError('Duplicate planned query geometry')
    if not set(plan.loc[plan.family=='new_groups','cell']).isdisjoint(old_cells):
        raise ValueError('Primary group leaked from old query inventory')
    for seed in seeds:
        assigned=membership[(membership.seed==seed)&(membership.family=='existing_test_groups')]
        test=set(outside.loc[(outside.seed==seed)&(outside.split=='test'),'cell'])
        other=set(outside.loc[(outside.seed==seed)&outside.split.isin(['train','calibration']),'cell'])
        if not set(assigned.cell)<=test or set(assigned.cell)&other:
            raise ValueError('Secondary membership overlaps model fitting/calibration groups')
    audit=dict(protocol=GENERATION_VERSION,query_seed=query_seed,maximum_diagonal_shift=MAX_DIAGONAL_SHIFT,
        near_geometry_tolerance=NEAR_GEOMETRY_TOL,all_old_query_count=len(q),all_old_cells=len(old_cells),
        possible_source_cells=len(possible),old_cells_outside_inventory=len(old_cells-possible),
        new_groups=len(primary),secondary_counts={str(s):len(selected_secondary[s]) for s in seeds},
        secondary_union_cells=len(secondary),planned_queries=len(plan),planned_model_query_cases=len(membership),
        geometries_preallocated=len(generated),rejected_near_old=rejected_near_old,
        rejected_duplicate_geometry=rejected_duplicate,geometry_selection_uses_outcomes=False,
        primary_old_group_overlap=0,secondary_fit_calibration_overlap=0,
        old_family_geometrymatch=_old_family_diagnostics(q,old_cell_array,grid_by_id),
        scope='new rectangles and generating groups on the same raw dataset; not strict geographic separation')
    return plan,membership,audit

## Build exact new truth from raw points

One raw source is loaded/indexed/released at a time. The spatial tree is exact: wholly enclosed nodes contribute their
original row count; otherwise leaf points are checked with inclusive <=/>= comparisons. Duplicates remain separate rows.
The index does not estimate from a grid and does not read saved ground truth. Its inputs are raw coordinates and rectangles.
Thirty-two fixed hash-selected new rectangles are also counted by independent brute-force raw masks; mismatches stop the run.

Raw hashes/row counts must match the original frozen inputs. `protocol_lock.json`, `query_plan.csv` and memberships are
written before counting and rechecked afterward. Counts<10 are excluded without regeneration or replacement. Labeling,
raw loading, index construction and brute auditing costs are recorded separately from resident query runtime.

In [ ]:
"""Offline exact 2D closed-rectangle counts directly over raw points.

The spatial tree is only an index: counts are either the number of original
rows wholly enclosed by a query or direct comparisons of original leaf points.
No histogram/label/model value enters the answer. Duplicate points count once
per raw row. One source is loaded/indexed/released at a time.
"""
from pathlib import Path
import time
import numpy as np
from scipy.spatial import cKDTree


def _exact_rectangles(rectangles):
    q = np.asarray(rectangles, dtype=np.float64)
    if q.ndim != 2 or q.shape[1] != 4 or not np.isfinite(q).all():
        raise ValueError('Rectangles must be finite [min_x,max_x,min_y,max_y] rows')
    if (q[:, 0] > q[:, 1]).any() or (q[:, 2] > q[:, 3]).any():
        raise ValueError('Rectangle lower bounds exceed upper bounds')
    return q


def exact_rectangle_count(tree, rectangle):
    """Count one closed rectangle using public cKDTree node attributes.

    Root bounds and split coordinates are existing float64 point coordinates.
    All comparisons use <=/>= directly; no epsilon expansion/normalization or
    approximate distance search changes the raw query semantics.
    """
    x0, x1, y0, y1 = map(float, rectangle)
    if tree.n == 0:
        return 0
    pending = [(tree.tree, float(tree.mins[0]), float(tree.maxes[0]),
                float(tree.mins[1]), float(tree.maxes[1]))]
    answer = 0
    while pending:
        node, lx, hx, ly, hy = pending.pop()
        # Strict exclusion preserves points on all four inclusive query edges.
        if hx < x0 or lx > x1 or hy < y0 or ly > y1:
            continue
        if x0 <= lx and hx <= x1 and y0 <= ly and hy <= y1:
            answer += int(node.children)
            continue
        if node.split_dim < 0:
            points = tree.data[tree.indices[node.start_idx:node.end_idx]]
            answer += int(np.count_nonzero((points[:, 0] >= x0) & (points[:, 0] <= x1) &
                                           (points[:, 1] >= y0) & (points[:, 1] <= y1)))
        elif node.split_dim == 0:
            split = float(node.split)
            pending.append((node.lesser, lx, split, ly, hy))
            pending.append((node.greater, split, hx, ly, hy))
        elif node.split_dim == 1:
            split = float(node.split)
            pending.append((node.lesser, lx, hx, ly, split))
            pending.append((node.greater, lx, hx, split, hy))
        else:
            raise ValueError('Expected a two-dimensional point tree')
    return answer


def count_point_rectangles(points, rectangles, leafsize=128):
    """Index raw coordinates and count each rectangle, returning setup/query cost."""
    points = np.asarray(points, dtype=np.float64, order='C')
    rectangles = _exact_rectangles(rectangles)
    if points.ndim != 2 or points.shape[1] != 2 or not np.isfinite(points).all():
        raise ValueError('Points must be finite two-dimensional coordinates')
    if int(leafsize) != leafsize or leafsize < 1:
        raise ValueError('leafsize must be positive')
    tick = time.perf_counter()
    tree = cKDTree(points, leafsize=int(leafsize), compact_nodes=True, balanced_tree=True, copy_data=False)
    # Materialize the documented Python tree view during initialization, not query timing.
    tree.tree
    build_seconds = time.perf_counter() - tick
    tick = time.perf_counter()
    counts = np.fromiter((exact_rectangle_count(tree, q) for q in rectangles),
                         dtype=np.int64, count=len(rectangles))
    query_seconds = time.perf_counter() - tick
    return counts, dict(raw_points=len(points), query_count=len(rectangles),
                       index_build_seconds=build_seconds, exact_query_seconds=query_seconds,
                       raw_coordinate_bytes=points.nbytes, leafsize=int(leafsize),
                       algorithm='exact closed 2D range count over cKDTree nodes/raw leaf points')


def exact_range_counts(points, rectangles, leafsize=128):
    """Return int64 exact inclusive raw-point counts; no labels or histograms.

    Zero-width/height closed boxes are well-defined and count boundary points.
    The new-query generator can impose its stricter positive-area requirement.
    """
    return count_point_rectangles(points, rectangles, leafsize=leafsize)[0]


def direct_point_counts(points, rectangles):
    """Independent brute-force oracle, no index or precomputed answers."""
    points = np.asarray(points, dtype=np.float64)
    rectangles = _exact_rectangles(rectangles)
    return np.array([np.count_nonzero((points[:, 0] >= q[0]) & (points[:, 0] <= q[1]) &
                                     (points[:, 1] >= q[2]) & (points[:, 1] <= q[3]))
                     for q in rectangles], dtype=np.int64)


def count_raw_rectangles(raw_paths, rectangles, iter_points, expected_rows=None,
                         audit_indices=(), leafsize=128):
    """Build exact new labels from files, one source at a time.

    iter_points(path) must yield finite Nx2 raw arrays (e.g. verified notebook15
    TAB/gzip reader). expected_rows, when given, avoids a concatenate double copy
    and detects changed/truncated sources. File hashes belong to the caller's
    provenance checks. audit_indices are fixed independently of outcomes.

    Returns (total_counts, source_diagnostics, brute_audit_rows). No true labels,
    model scores, features or histogram answers are accepted.
    """
    rectangles = _exact_rectangles(rectangles)
    raw_paths = list(raw_paths)
    if expected_rows is not None and len(expected_rows) != len(raw_paths):
        raise ValueError('One expected row count is required per source')
    audit_indices = np.asarray(audit_indices, dtype=np.int64)
    if (audit_indices < 0).any() or (audit_indices >= len(rectangles)).any():
        raise ValueError('Audit query index outside new query set')
    if len(set(audit_indices.tolist())) != len(audit_indices):
        raise ValueError('Duplicate audit indices')
    total = np.zeros(len(rectangles), dtype=np.int64)
    brute_total = np.zeros(len(audit_indices), dtype=np.int64)
    diagnostics = []
    for source_index, path in enumerate(raw_paths):
        tick = time.perf_counter()
        if expected_rows is None:
            pieces = list(iter_points(path))
            points = np.concatenate(pieces, axis=0) if pieces else np.empty((0, 2), dtype=np.float64)
            del pieces
        else:
            n = int(expected_rows[source_index])
            if n < 0:
                raise ValueError('Expected source row count cannot be negative')
            points = np.empty((n, 2), dtype=np.float64)
            position = 0
            for part in iter_points(path):
                part = np.asarray(part, dtype=np.float64)
                if part.ndim != 2 or part.shape[1] != 2 or not np.isfinite(part).all():
                    raise ValueError('Raw reader emitted invalid coordinates')
                if position + len(part) > n:
                    raise ValueError('Raw source has more rows than its saved identity')
                points[position:position + len(part)] = part
                position += len(part)
            if position != n:
                raise ValueError('Raw source has fewer rows than its saved identity')
        load_seconds = time.perf_counter() - tick
        counts, info = count_point_rectangles(points, rectangles, leafsize=leafsize)
        total += counts
        tick = time.perf_counter()
        brute = direct_point_counts(points, rectangles[audit_indices])
        brute_total += brute
        if not np.array_equal(counts[audit_indices], brute):
            raise ValueError('Indexed exact counts disagree with independent raw-point brute scan')
        info.update(source_index=source_index, raw_path=str(path), raw_load_seconds=load_seconds,
                    brute_audit_seconds=time.perf_counter() - tick)
        diagnostics.append(info)
        del points
    rows = [dict(query_index=int(i), indexed_exact_count=int(total[i]),
                 brute_raw_count=int(v), matches=bool(total[i] == v))
            for i, v in zip(audit_indices, brute_total)]
    return total, diagnostics, rows



def freeze_plan(cfg,data,destination,state):
    source_ids=[int(r['source_id']) for r in data['raw_manifest']]
    plan,membership,audit=generate_plan(data['queries'][:,:4],data['coarse_hists'],source_ids,
        data['outer'],cfg['seeds'],query_seed=cfg['query_seed'],replication_cells=cfg['replication_cells'])
    if plan.empty:raise ValueError('No fresh query candidates can be generated')
    plan.to_csv(destination/'query_plan.csv',index=False,float_format='%.17g')
    membership.to_csv(destination/'query_membership.csv',index=False)
    save_json(destination/'generation_audit.json',audit)
    # Define every decision before accessing new truth. Changing query_seed after seeing results creates a new exploration.
    protocol=dict(version='fresh-fixed-H-v1',implementation_sha256=IMPLEMENTATION_SHA256,
        query_seed=cfg['query_seed'],replication_cells=cfg['replication_cells'],seeds=cfg['seeds'],methods=list(METHODS),
        primary='one diagonal-shift query in EVERY source generating cell absent from ALL original query rectangles',
        secondary='one new query per deterministic sampled ORIGINAL outer-test cell; evaluated only by its held-out model seed',
        generator='one-source-cell width/height; shared x/y shift U(0,sqrt(.05)); hash keyed by query_seed and cell',
        min_exact=MIN_EXACT,no_refill_after_truth=True,epsilon=EPSILON,fixed_coverages=list(FRACTIONS),
        training=False,model_selection=False,feature_selection=False,gate_retuning=False,acceptance_risk_target=None,
        proxy='abs(H-Sfine)/max(Sfine,1)',ml_prediction='max(saved raw-relative-error L1 booster output,0); no upper clip',
        labels='exact closed-rectangle counts over raw point rows; spatial index plus independent brute audit',
        raw_manifest_sha256=sha(Path(state['source15_run'])/'raw_input_manifest.json'),
        model_sha256={str(seed):sha(data['model_paths'][seed]) for seed in cfg['seeds']},
        query_plan_sha256=sha(destination/'query_plan.csv'),membership_sha256=sha(destination/'query_membership.csv'),
        claim_scope='same raw dataset; unseen generating IDs are not strict non-overlapping geography or a new dataset')
    save_json(destination/'protocol_lock.json',protocol)
    state.update(protocol_sha256=sha(destination/'protocol_lock.json'),query_plan_sha256=protocol['query_plan_sha256'],
                 query_membership_sha256=protocol['membership_sha256'],generation=audit,phase='geometry_frozen_before_new_truth')
    save_json(destination/'run.json',state)
    return plan,membership


def build_exact_labels(cfg,data,helpers,plan,membership,destination,state):
    raw_paths=[]
    for record in data['raw_manifest']:
        name=Path(record['raw_path']).name
        choices=[Path(cfg['raw_data_dir'])/name] if cfg['raw_data_dir'] else [Path(record['raw_path']),Path(cfg['repo_root'])/'range_query'/name]
        path=next((p for p in choices if p.is_file()),None)
        if path is None:raise FileNotFoundError('Missing matching raw file '+name+'. Set raw_data_dir to its unchanged folder')
        if sha(path)!=record['raw_sha256']:raise ValueError('Raw file differs from frozen15 identity: '+str(path))
        raw_paths.append(path.resolve())
    original_parity=pd.read_csv(Path(state['source15_run'])/'raw_source_parity.csv').set_index('source_id')
    expected=[int(original_parity.loc[int(r['source_id']),'raw_rows']) for r in data['raw_manifest']]
    audit_ids=stable_ids(plan.query_id,32,'17-direct-raw-audit-v1')
    audit_indices=np.flatnonzero(plan.query_id.isin(audit_ids))
    rects=plan[['x0','x1','y0','y1']].to_numpy(float)
    tick=time.perf_counter()
    truth,diagnostics,audits=count_raw_rectangles(raw_paths,rects,helpers['iter_raw_points'],expected_rows=expected,
                                                audit_indices=audit_indices,leafsize=128)
    state['timings']['exact_raw_label_construction_seconds']=time.perf_counter()-tick
    for path,record in zip(raw_paths,data['raw_manifest']):
        if sha(path)!=record['raw_sha256']:raise ValueError('Raw input changed during exact labeling')
    pd.DataFrame(diagnostics).to_csv(destination/'exact_label_costs.csv',index=False)
    audit=pd.DataFrame(audits);audit['query_id']=plan.iloc[audit.query_index].query_id.to_numpy()
    audit.to_csv(destination/'exact_label_audit.csv',index=False)
    if not audit.matches.all():raise ValueError('Independent direct raw scans disagree with exact spatial-index counts')
    if sha(destination/'query_plan.csv')!=state['query_plan_sha256'] or sha(destination/'query_membership.csv')!=state['query_membership_sha256']:
        raise ValueError('Frozen geometry/membership changed after truth access')
    labelled=plan.assign(truth=truth,eligible=truth>=MIN_EXACT,zero_truth=truth==0)
    labelled.to_csv(destination/'new_queries.csv',index=False,float_format='%.17g')
    labelled[~labelled.eligible].assign(reason='truth below inherited min_exact=10; not replaced').to_csv(destination/'excluded_queries.csv',index=False)
    counts=membership.merge(labelled[['query_id','truth','eligible','zero_truth']],on='query_id',validate='many_to_one')
    rows=[]
    for (family,seed),part in counts.groupby(['family','seed']):
        rows.append(dict(family=family,seed=seed,planned_queries=len(part),eligible_queries=int(part.eligible.sum()),
                         excluded_below10=int((~part.eligible).sum()),zero_truth=int(part.zero_truth.sum()),
                         unique_cells=part.cell.nunique(),no_outcome_refill=True))
    pd.DataFrame(rows).to_csv(destination/'cohort_counts.csv',index=False)
    state.update(unique_generated_queries=len(plan),unique_eligible_queries=int(labelled.eligible.sum()),
                 excluded_small_truth=int((~labelled.eligible).sum()),zero_truth_candidates=int(labelled.zero_truth.sum()),
                 new_query_labels_sha256=sha(destination/'new_queries.csv'),phase='new_truth_created_without_refill')
    save_json(destination/'run.json',state)
    return labelled

## Evaluate all eligible queries, not only the timing sample

`cohort_counts.csv` reports planned/eligible/zero/excluded counts by family and model seed. The primary and secondary
cohorts remain separate throughout. SA and proxy have no fitted model; repeating them across model seeds is not additional
independent evidence. Native model scores use the saved feature order and fixed zero-floor, with no100% upper clip.
The count-only path and frozen feature provider must agree on both H and Sfine.

In [ ]:
def prepare_live_contexts(cfg,data,namespace,state):
    import sys
    sys.path.insert(0,str(Path(cfg['repo_root'])/'ml'))
    import features
    if Path(features.__file__).resolve().parent!=Path(cfg['repo_root']).resolve()/'ml':raise ValueError('Another checkout is imported; restart Python')
    tick=time.perf_counter()
    contexts=dict(base={r:namespace['histogram_context'](h) for r,h in data['base_hists'].items()},
                  sources=[namespace['histogram_context'](h) for h in data['raw_hists']],merged_helpers=features)
    state['initialization'].append(dict(method='raw2_ml',kind='resident_numpy_feature_contexts',seconds=time.perf_counter()-tick))
    runtime={k:data[k] for k in ('base_hists','alphas','raw_hists','coarse_hists','models','names')}
    runtime['num_threads']=state['num_threads']
    return contexts,runtime


def evaluate_fresh(cfg,data,env,helpers,namespace,labelled,membership,destination,state,contexts,runtime):
    eligible=labelled[labelled.eligible].set_index('query_id')
    if eligible.empty:raise ValueError('No generated query meets min_exact=10; no replacement sampling is allowed')
    by_id={q:row[['x0','x1','y0','y1']].to_numpy(float) for q,row in eligible.iterrows()}
    metrics=[];predictions=[];coverage=[];curves=[];aux=[];audits=[];empty=[];costs=[]
    memberships=membership[membership.query_id.isin(eligible.index)].copy()
    for rate in data['rates']:
        qids=sorted(eligible.index);h=[];sa=[];sf=[];matrix=[]
        for qid in qids:
            rect=by_id[qid]
            answer,score=env['query_answer_sa'](rect,data['base_hists'][rate],data['alphas'][rate])
            source=sum(env['histogram_count'](rect,s) for s in data['raw_hists'])
            f=namespace['extract_query'](rect,contexts['base'][rate],contexts['merged_helpers'],sources=contexts['sources'])
            if not np.isclose(f['est_py'],answer,rtol=1e-10,atol=1e-8) or not np.isclose(np.expm1(f['src_log_est']),source,rtol=1e-10,atol=1e-8):
                raise ValueError('Frozen feature provider and count-only estimators disagree on new query')
            f.update({'raw2_'+k:v for k,v in list(f.items()) if k.startswith('src_')})
            matrix.append([f[n] for n in data['names']]);h.append(answer);sa.append(score);sf.append(source)
        x=pd.DataFrame(matrix,index=qids,columns=data['names'])
        if not np.isfinite(x.to_numpy()).all():raise ValueError('Nonfinite frozen features on fresh queries')
        physical=pd.DataFrame(dict(query_id=qids,answer=h,sa_score=sa,s_fine=sf)).set_index('query_id')
        physical['proxy']=[env['proxy_error'](a,s) for a,s in zip(physical.answer,physical.s_fine)]
        physical['s_fine_zero']=physical.s_fine==0;physical['base_rate']=rate
        aux.append(physical.reset_index())
        for seed in cfg['seeds']:
            ids=sorted(set(memberships.loc[memberships.seed==seed,'query_id']))
            if not ids:continue
            native=data['models'][seed].predict(x.loc[ids],num_threads=state['num_threads'])
            if not np.isfinite(native).all():raise ValueError('Nonfinite frozen-model output')
            ml=pd.Series(np.maximum(native,0.),index=ids)
            for family in FAMILIES:
                selected=sorted(set(memberships.loc[(memberships.seed==seed)&(memberships.family==family),'query_id']))
                if not selected:
                    empty.append(dict(family=family,seed=seed,base_rate=rate,reason='no eligible predeclared queries; no refill'));continue
                part=physical.loc[selected];truth=eligible.loc[selected,'truth'].to_numpy(float)
                err=abs(part.answer.to_numpy()-truth)/truth
                for method,score,pred in [('base_sa',part.sa_score.to_numpy(),None),
                        ('raw2_proxy',part.proxy.to_numpy(),part.proxy.to_numpy()),('raw2_ml',ml.loc[selected].to_numpy(),ml.loc[selected].to_numpy())]:
                    prefix=dict(family=family,seed=seed,base_rate=rate,method=method)
                    metrics.append(dict(**prefix,**helpers['prediction_metrics'](err,pred,score)))
                    record=pd.DataFrame(dict(query_id=selected,truth=truth,answer=part.answer.to_numpy(),actual_rel_error=err,
                        score=score,predicted_error=np.nan if pred is None else pred,**prefix))
                    record=record.merge(eligible[['cell','source_id']],left_on='query_id',right_index=True,validate='many_to_one')
                    predictions.append(record)
                    coverage.extend(dict(**prefix,**r) for r in env['fixed_coverage'](err,score))
                    curves.extend(dict(**prefix,**r) for r in helpers['risk_coverage_rows'](err,score))
                audits.append(dict(family=family,seed=seed,base_rate=rate,queries=len(selected),
                    all_fixed_answer_targets_match=True,no_old_query_duplicates=True,
                    generating_group_status='absent from ALL old query groups' if family=='new_groups' else 'old outer-test groups only'))
        for seed in cfg['seeds']:
            base_bytes=data['base_paths'][rate].stat().st_size;raw_bytes=sum(p.stat().st_size for p in data['raw_paths'])
            for method in METHODS:
                synopsis=base_bytes+8 if method=='base_sa' else base_bytes+raw_bytes
                model=data['model_paths'][seed].stat().st_size if method=='raw2_ml' else 0
                costs.append(dict(seed=seed,base_rate=rate,method=method,synopsis_bytes=synopsis,model_bytes=model,
                                  total_serialized_bytes=synopsis+model,process_memory_included=False))
    if not predictions:raise ValueError('No eligible evaluation settings')
    result=pd.concat(predictions,ignore_index=True)
    if result.duplicated(['family','seed','base_rate','method','query_id']).any():raise ValueError('Duplicate fresh prediction keys')
    result.to_csv(destination/'per_query.csv',index=False)
    metric=pd.DataFrame(metrics);metric.to_csv(destination/'metrics_by_setting.csv',index=False)
    pd.DataFrame(coverage).to_csv(destination/'fixed_coverage.csv',index=False)
    pd.DataFrame(curves).to_csv(destination/'risk_coverage.csv',index=False)
    pd.concat(aux,ignore_index=True).to_csv(destination/'auxiliary_estimates.csv',index=False)
    pd.DataFrame(audits).to_csv(destination/'cohort_audit.csv',index=False)
    pd.DataFrame(empty,columns=['family','seed','base_rate','reason']).to_csv(destination/'empty_settings.csv',index=False)
    pd.DataFrame(costs).to_csv(destination/'payload_costs.csv',index=False)
    metric.groupby(['family','method','base_rate'],as_index=False).agg(seeds=('seed','nunique'),
        mean_mae_pp=('mae_pp','mean'),mean_p90_ae_pp=('p90_ae_pp','mean'),mean_bias_pp=('bias_pp','mean'),
        mean_auc_bad=('auc_bad','mean'),mean_bad_fraction=('bad_fraction','mean')).to_csv(destination/'summary.csv',index=False)
    return result,by_id

## Paired evidence and live cost

ML-minus-proxy MAE differences first average repeated model appearances for each query, then resample generating groups.
Intervals are conditional/descriptive; different groups can overlap geographically. These intervals do not certify a
population risk or replace a later genuinely external dataset test.

Runtime providers receive only rectangle coordinates and resident histograms/models/feature contexts. They receive no
truth, label table, old predictions or raw paths. All three methods share timing queries, one fixed warm-up and randomized
paired arm order. The timer ends before saved-vs-live checks. Initialization and serialized bytes are reported separately.

In [ ]:
def paired_effects(predictions,destination):
    rows=[]
    for (family,rate),part in predictions.groupby(['family','base_rate']):
        a=part[part.method=='raw2_ml'];b=part[part.method=='raw2_proxy']
        pairs=a.merge(b,on=['seed','query_id'],suffixes=('_ml','_proxy'),validate='one_to_one')
        if not np.array_equal(pairs.actual_rel_error_ml,pairs.actual_rel_error_proxy):raise ValueError('Paired models have different targets')
        pairs['delta']=100*(abs(pairs.predicted_error_ml-pairs.actual_rel_error_ml)-abs(pairs.predicted_error_proxy-pairs.actual_rel_error_proxy))
        perq=pairs.groupby('query_id').agg(delta=('delta','mean'),cell=('cell_ml','first'))
        groups=perq.groupby('cell').delta.agg(['sum','count']);rng=np.random.default_rng(17001);boot=[]
        if len(groups)>1:
            for _ in range(2000):
                ix=rng.integers(0,len(groups),len(groups));boot.append(float(groups['sum'].to_numpy()[ix].sum()/groups['count'].to_numpy()[ix].sum()))
        rows.append(dict(family=family,base_rate=rate,unique_queries=len(perq),generating_groups=len(groups),
            mean_ml_minus_proxy_mae_pp=float(perq.delta.mean()),ci95_low_pp=float(np.quantile(boot,.025)) if boot else np.nan,
            ci95_high_pp=float(np.quantile(boot,.975)) if boot else np.nan,bootstrap_repeats=len(boot),
            uncertainty_scope='conditional descriptive generating-group resampling; repeated model-seed appearances averaged per query; groups can overlap geographically'))
    pd.DataFrame(rows).to_csv(destination/'paired_ml_vs_proxy.csv',index=False)


def time_fresh(cfg,data,env,namespace,contexts,runtime,predictions,rectangles,destination,state):
    expected=predictions.set_index(['family','seed','base_rate','method','query_id'])[['answer','score']]
    records=[];audits=[]
    for family in FAMILIES:
        for seed in cfg['seeds']:
            available=predictions.loc[(predictions.family==family)&(predictions.seed==seed),'query_id']
            selected=stable_ids(available,cfg['timing_queries'],f'17-timing:{family}:{seed}')
            if not selected:continue
            for rate in data['rates']:
                for qid in selected:
                    for method in METHODS:env['runtime_call'](rectangles[qid],method,rate,seed,runtime,namespace,contexts,{},None)
                rng=np.random.default_rng(170000+seed)
                for repeat in range(cfg['timing_repeats']):
                    for qid in rng.permutation(selected):
                        rect=rectangles[qid]
                        for order,method in enumerate(rng.permutation(METHODS)):
                            start=time.perf_counter_ns()
                            answer,score=env['runtime_call'](rect,method,rate,seed,runtime,namespace,contexts,{},None)
                            stop=time.perf_counter_ns()
                            reference=expected.loc[(family,seed,rate,method,qid)]
                            if not np.isclose(answer,reference.answer,rtol=1e-10,atol=1e-8) or not np.isclose(score,reference.score,rtol=1e-9,atol=1e-8):
                                raise ValueError('Live fresh-query score differs from batch')
                            records.append(dict(family=family,seed=seed,base_rate=rate,query_id=qid,method=method,
                                repeat=repeat,arm_order=order,total_ms=(stop-start)/1e6,
                                cache='resident histograms/models/contexts; one warm-up/query/arm; randomized paired order; no raw reads'))
                audits.extend(dict(family=family,seed=seed,base_rate=rate,method=m,live_answer_and_score_match=True) for m in METHODS)
    timing=pd.DataFrame(records);timing.to_csv(destination/'resident_timings.csv',index=False)
    timing.groupby(['family','seed','base_rate','method'],as_index=False).agg(measured_requests=('total_ms','size'),
        unique_queries=('query_id','nunique'),mean_ms=('total_ms','mean'),p95_ms=('total_ms',lambda s:s.quantile(.95))).to_csv(destination/'resident_timing_summary.csv',index=False)
    pd.DataFrame(audits).to_csv(destination/'live_parity.csv',index=False)
    pd.DataFrame(state['initialization']).to_csv(destination/'initialization.csv',index=False)

## Read and share

The ZIP contains fresh query geometry/labels, memberships, frozen protocol hashes, audits, complete per-query predictions,
summary/coverage/paired CSVs, costs and three English figures. Raw points, models and payloads are excluded.
Do not treat a rerun with identical salt/inputs as an independent new experiment. If a result is weak, retain it rather
than changing the query seed or denominator until the result looks favorable.

In [ ]:
def write_report(destination,state):
    import matplotlib.pyplot as plt
    summary=pd.read_csv(destination/'summary.csv');coverage=pd.read_csv(destination/'fixed_coverage.csv')
    timing=pd.read_csv(destination/'resident_timing_summary.csv');costs=pd.read_csv(destination/'payload_costs.csv')
    labels={'new_groups':'Primary: previously unused generating groups','existing_test_groups':'Supplement: new queries in old test groups'}
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for i,family in enumerate(FAMILIES):
        part=summary[summary.family==family]
        if part.empty:
            for ax in axes[i]:ax.text(.5,.5,'No eligible queries; no refill',ha='center');ax.set_axis_off()
            continue
        a=part.groupby('method').mean_mae_pp.mean().reindex(['raw2_proxy','raw2_ml'])
        b=part.groupby('method').mean_auc_bad.mean().reindex(METHODS)
        axes[i,0].bar(a.index,a);axes[i,0].set(ylabel='Error-prediction MAE (pp)',title=labels[family])
        axes[i,1].bar(b.index,b);axes[i,1].set(ylabel='AUC for actual error >25%',ylim=(0,1),title='Same fixed histogram answer')
    fig.tight_layout();fig.savefig(destination/'prediction_quality.png',dpi=150);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,family in zip(axes,FAMILIES):
        for method in METHODS:
            part=coverage[(coverage.family==family)&(coverage.method==method)]
            if part.empty:continue
            line=part.groupby('nominal_coverage')[['actual_coverage','violation']].mean()
            ax.plot(line.actual_coverage,line.violation,marker='o',label=method)
        ax.set(xlabel='Actual selected fraction (whole ties)',ylabel='Violation fraction among selected answers',title=labels[family]);ax.legend()
    fig.tight_layout();fig.savefig(destination/'risk_coverage.png',dpi=150);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    a=costs.groupby('method').total_serialized_bytes.mean().reindex(METHODS)/1024
    axes[0].bar(a.index,a);axes[0].set_ylabel('Synopsis + model bytes (KiB)')
    for family in FAMILIES:
        part=timing[timing.family==family]
        if part.empty:continue
        a=part.groupby('method').mean_ms.mean().reindex(METHODS)
        axes[1].plot(a.index,a,marker='o',label=family)
    axes[1].set_ylabel('Mean resident request time (ms)');axes[1].legend()
    for ax in axes:ax.tick_params(axis='x',rotation=20)
    fig.tight_layout();fig.savefig(destination/'costs.png',dpi=150);plt.close(fig)
    text='''# Frozen evaluator validation on fresh queries

The same original histogram answer H is scored by faithful SA, the fixed raw2 disagreement proxy, and saved raw2 ML. No model is trained, no feature/formula is changed, and no operational gate is calibrated. Numerical SA error MAE is undefined. A 25% actual-error tolerance defines ranking labels, not a 5% accepted-risk target.

## Predeclared cohorts

- **new_groups (primary):** one new diagonal-shift query for EVERY source generating-cell ID absent from ALL original query rectangles, including original low/zero-count queries. All selected frozen model seeds see the same primary query pool; these IDs never appear in any original training/calibration/test group.
- **existing_test_groups (supplement):** at most the configured number of cells per original outer-test split, chosen by fixed hash, one new query per cell. Each model is evaluated only on its own held-out test groups. These groups were already inspected in earlier analyses.
- The cohorts remain separate. Repeated seeds do not create independent query samples. Geometry is deterministic per cell/salt and excludes old rectangles; its plan, membership and protocol hashes are written BEFORE accessing new truth. Candidates below the inherited exact-count minimum of 10 are excluded without replacement. Zero-count query performance is not claimed.
- The source grids and raw dataset are unchanged. Generating IDs from different source grids can overlap geographically; this is neither a new dataset nor strict non-overlapping geographic transfer. The new one-cell diagonal-shift family is explicitly defined; generation_audit.json records how it relates to observed old geometry, without claiming an unavailable original generator was reproduced bit-for-bit.

## Results by cohort

'''+summary.round(6).to_csv(index=False)+'''
MAE is the mean difference between predicted and actual relative error, in percentage points, not classification accuracy or the histogram answer's own count error. Prediction tail error and bias must be read with average MAE. Higher AUC does not guarantee a better lowest-risk tail; fixed_coverage.csv reports actual counts, complete ties and observed violations at 10/25/50/100% diagnostic fractions. No diagnostic cutoff is deployed or selected using truth.

## Exact labels and cost

Truth is newly computed from the matching original raw rows through an exact closed-rectangle spatial index. Full-contained nodes contribute their original row count; partial leaves are filtered directly. Duplicates remain separate rows. This is not histogram estimation or lookup of old truth. Thirty-two fixed hash-selected new rectangles are independently checked by direct raw scans. Raw identities and row counts must match the frozen input. Exact labeling/indexing is offline validation cost, outside query timing.

Resident query timing performs real H estimation and SA/proxy/inference, using the same queries with paired randomized arm order and fixed warm-up. Runtime providers receive no truth/old predictions/raw files. Batch-vs-live parity is checked after each timer. Model loading/context initialization and payload/model bytes are separate; no remote I/O, exact fallback or E2E speedup is measured.

## Interpretation limits

Compare the two cohorts separately and retain failed/missing settings. A weak or empty primary cohort is not replaced with an easier cohort. paired_ml_vs_proxy.csv first averages repeated model appearances per query and resamples generating groups, conditional on the fitted models; groups may overlap geographically, so it is a descriptive sensitivity interval rather than a population guarantee.

This validates a candidate selected after earlier experiments on new labels within the same underlying dataset. It does not erase all prior study-selection effects or establish broad generalization. Do not change salt, formula, model or thresholds after seeing these outcomes and then call a rerun a new independent confirmation. If no advantage persists, report it and stop that candidate. If an advantage persists, decide which fixed-quality/cost use case matters before a later selected-method E2E experiment. The finer-answer alternative from 16 remains relevant; this notebook deliberately fixes H rather than claiming raw2 is the best answer representation.
'''
    (destination/'analysis_en.md').write_text(text)


def export_results(destination):
    with ZipFile(destination/'fresh_query_validation_results.zip','w',ZIP_DEFLATED) as archive:
        for path in sorted(destination.iterdir()):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'}:archive.write(path,path.name)


def run_fresh_validation(cfg):
    global lgb,uniform_filter,roc_auc_score
    import lightgbm as lgb
    from scipy.ndimage import uniform_filter
    from sklearn.metrics import roc_auc_score
    destination=Path(cfg['output_root']).expanduser().resolve()/('fresh-query-validation-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state=dict(status='running',study='fresh-query-validation',config=cfg,initialization=[],timings={},
        started_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),epsilon=EPSILON,
        training=False,gate_retuning=False,acceptance_risk_target=None,
        scope='new generating IDs primary; fresh rectangles in existing held-out groups supplementary; same raw dataset')
    save_json(destination/'run.json',state);started=time.perf_counter()
    print('Writing new fresh-query validation:',destination,flush=True)
    try:
        source,prior=resolve_source(cfg)
        tick=time.perf_counter()
        data,env,helpers,namespace,prior15=load_frozen_inputs(cfg,source,prior,destination,state)
        state['timings']['frozen_artifact_validation_and_loading_seconds']=time.perf_counter()-tick
        plan,membership=freeze_plan(cfg,data,destination,state)
        print('Frozen candidate queries:',len(plan),'memberships:',len(membership),flush=True)
        labelled=build_exact_labels(cfg,data,helpers,plan,membership,destination,state)
        contexts,runtime=prepare_live_contexts(cfg,data,namespace,state)
        tick=time.perf_counter()
        predictions,rectangles=evaluate_fresh(cfg,data,env,helpers,namespace,labelled,membership,destination,state,contexts,runtime)
        state['timings']['fresh_full_cohort_evaluation_seconds']=time.perf_counter()-tick
        paired_effects(predictions,destination)
        time_fresh(cfg,data,env,namespace,contexts,runtime,predictions,rectangles,destination,state)
        write_report(destination,state)
        for filename,key in [('query_plan.csv','query_plan_sha256'),('query_membership.csv','query_membership_sha256'),('protocol_lock.json','protocol_sha256')]:
            if sha(destination/filename)!=state[key]:raise ValueError('Frozen protocol artifact changed: '+filename)
        state.update(status='complete',completed_utc=datetime.now(timezone.utc).isoformat(),
            total_seconds=time.perf_counter()-started,prediction_rows=len(predictions))
        save_json(destination/'run.json',state);export_results(destination)
        print('Completed:',destination,flush=True)
        print('Share:',destination/'fresh_query_validation_results.zip',flush=True)
        return destination
    except Exception as error:
        state.update(status='failed',error_type=type(error).__name__,error=str(error),total_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state);export_results(destination)
        print('Failed run diagnostics:',destination/'fresh_query_validation_results.zip',flush=True)
        raise

## Execute the frozen validation

No model is trained. The primary query pool is all globally unused source groups; the supplement is sampled only from
held-out groups. Inspect `protocol_lock.json`, `generation_audit.json`, `exact_label_audit.csv`, `cohort_counts.csv` and
`live_parity.csv` before interpreting metrics. Share **`fresh_query_validation_results.zip`**, including failed-run
output if the run stops. An empty/weak primary cohort is not replaced with the supplement.

In [ ]:
# RUN_FRESH_VALIDATION
config = load_settings()
result_dir = run_fresh_validation(config)
print(pd.read_csv(result_dir / 'cohort_counts.csv').to_string(index=False))
print(pd.read_csv(result_dir / 'summary.csv').to_string(index=False))